# Aggregated cost ratios

This notebook reads `result_2_*.json` (aggregate ratios) and plots performance ratios across LLMs for selected costs.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

llms = ['gemma2_9b', 'llama3.1_8b', 'mistral_7b', 'qwen2.5_7b']
rms = ['fsfairx_rm', 'mistral_rm']

llm_mapping = {
    'gemma2_9b': 'google/gemma-2-9b-it',
    'llama3.1_8b': 'meta-llama/Llama-3.1-8B-Instruct',
    'llama3.2_3b': 'meta-llama/Llama-3.2-3B-Instruct',
    'mistral_7b': 'mistralai/Mistral-7B-Instruct-v0.3',
    'qwen2.5_7b': 'Qwen/Qwen2.5-7B-Instruct',
    'qwen3_4b': 'Qwen/Qwen3-4B-Instruct-2507',
}

rm_mapping = {
    'fsfairx_rm': 'sfairXC/FsfairX-LLaMA3-RM-v0.1',
    'mistral_rm': 'weqweasdas/RM-Mistral-7B',
}


In [ ]:
# Config
dataset = 'alpaca'
rm = 'fsfairx_rm'
distribution = 'shifted_exponential'
transformation = 'cdf'
batch_size = '1'
alpha = '0.99'

# Pick cost indices from data['costs']
cost_indices = [0, 3, 6, 9]  # adjust as needed


In [ ]:
def _load_ratios_for_llm(llm_name):
    path = Path(f'../slurm_result_{dataset}/result_2_{rm}_{llm_name}_{distribution}_{transformation}_bs{batch_size}_a{alpha}.json')
    with path.open() as f:
        data = json.load(f)

    costs = data['costs']
    ratios = data['ratios']

    medians = [ratios[i]['median_ratio'] for i in cost_indices]
    p25 = [ratios[i]['p25_ratio'] for i in cost_indices]
    p75 = [ratios[i]['p75_ratio'] for i in cost_indices]
    costs_shortlist = [(i, costs[i]) for i in cost_indices]

    return costs_shortlist, {
        'median_ratio': np.array(medians, dtype=float),
        'p25_ratio': np.array(p25, dtype=float),
        'p75_ratio': np.array(p75, dtype=float),
    }


In [ ]:
def build_ratios_by_llm():
    ratios_per_llm = {}
    costs_shortlist = None
    for llm_name in llms:
        costs_shortlist, ratios = _load_ratios_for_llm(llm_name)
        ratios_per_llm[llm_name] = ratios
    return costs_shortlist, ratios_per_llm


In [ ]:
def plot_ratios_barchart(ratios_per_llm, costs_shortlist, rm_name, dataset_name):
    llm_labels = list(ratios_per_llm.keys())
    n_llms = len(llm_labels)
    n_costs = len(costs_shortlist)

    x = np.arange(n_costs)
    width = 0.15

    fig, ax = plt.subplots(figsize=(16, 4))

    for i, llm_name in enumerate(llm_labels):
        ratios = ratios_per_llm[llm_name]
        q25, medians, q75 = ratios['p25_ratio'], ratios['median_ratio'], ratios['p75_ratio']
        yerr = np.vstack([medians - q25, q75 - medians])
        offset = width * (i - (n_llms - 1) / 2)
        ax.bar(
            x + offset,
            medians,
            width,
            label=llm_mapping.get(llm_name, llm_name),
            yerr=yerr,
            capsize=4,
            ecolor='black',
        )
        ax.scatter(x + offset, medians, color='black', s=25, zorder=3)

    ax.set_ylabel('Profit Performance Ratio (IQR)', fontsize=16)
    dataset_label = 'AlpacaEval' if dataset_name == 'alpaca' else 'HH-RLHF'
    ax.set_title(
        f'Reward Model: {rm_mapping.get(rm_name, rm_name)} | Dataset: {dataset_label}',
        fontsize=16,
    )
    cost_labels = [f'Cost = {c}' for _, c in costs_shortlist]
    ax.set_ylim([0, 1.2])
    ax.set_xticks(x, cost_labels, fontsize=16)
    ax.tick_params(axis='y', labelsize=16)
    ax.legend(title='LLM Models', bbox_to_anchor=(1.04, 1), loc='upper left', fontsize=16)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    ax.axhline(y=1, color='grey', linestyle='--', linewidth=1.5, label='Baseline (Non-Adaptive)')

    fig.tight_layout()
    plt.show()
    return fig


In [ ]:
costs_shortlist, ratios_per_llm = build_ratios_by_llm()
plot_ratios_barchart(ratios_per_llm, costs_shortlist, rm_name=rm, dataset_name=dataset)
